# Sheet 0.3: Probability, statistics and information theory

**Roadmap topic 0.3.** Read first: Stat 110 lectures, MML ch. 6, Olah's *Visual Information Theory*, Schulman's *Approximating KL Divergence*.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells, answer the **Written** questions before reading the hints at the bottom. CPU is fine. `scipy` is preinstalled on Colab.

In [ ]:
import math, torch
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

## Exercise 1: Entropy, cross-entropy and KL with the edge cases right

Implement for discrete distributions given as probability tensors on the last dimension (batched):
- `entropy(p)`, `cross_entropy(p, q)`, `kl(p, q)` in nats.
- Conventions: `0·log 0 = 0`. If `q = 0` where `p > 0`, cross-entropy and KL are `+inf`. No `nan` ever.
- `ce_from_logits(p, logits)`: cross-entropy of target distribution `p` against `softmax(logits)`, stable for logits up to ±1e4 (use log-softmax, not softmax then log).
- `label_smoothed_ce(logits, target, eps)`: standard label smoothing where the target distribution is `(1-eps)·onehot + eps/V`.

In [ ]:
def entropy(p):
    # YOUR CODE
    raise NotImplementedError
def cross_entropy(p, q):
    # YOUR CODE
    raise NotImplementedError
def kl(p, q):
    # YOUR CODE
    raise NotImplementedError
def ce_from_logits(p, logits):
    # YOUR CODE
    raise NotImplementedError
def label_smoothed_ce(logits, target, eps):
    # logits (B,V), target (B,) int64 -> mean loss over batch
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    p = torch.softmax(torch.randn(64, 10) * 2, -1); q = torch.softmax(torch.randn(64, 10) * 2, -1)
    assert torch.allclose(cross_entropy(p, q), entropy(p) + kl(p, q)), "CE != H + KL"
    assert (kl(p, q) >= 0).all() and torch.allclose(kl(p, p), torch.zeros(64), atol=1e-12)
    p0 = torch.tensor([0.5, 0.5, 0.0]); q0 = torch.tensor([0.5, 0.0, 0.5])
    assert abs(entropy(p0) - math.log(2)) < 1e-12 and not torch.isnan(entropy(p0))
    assert torch.isinf(kl(p0, q0)) and torch.isinf(cross_entropy(p0, q0)), "q=0 where p>0 must give inf"
    assert torch.isfinite(kl(q0 * 0 + torch.tensor([1., 0., 0.]), torch.tensor([.5, .5, 0.]))), "p=0, q=0 terms must be 0"
    big = torch.tensor([[1e4, -1e4, 0.0]]); tgt = torch.tensor([[0.2, 0.3, 0.5]])
    ref = -(tgt * torch.log_softmax(big, -1)).sum()
    assert torch.isfinite(ce_from_logits(tgt, big)).all() and torch.allclose(ce_from_logits(tgt, big), ref.reshape(1))
    logits = torch.randn(32, 50); t = torch.randint(0, 50, (32,))
    ref = torch.nn.functional.cross_entropy(logits, t, label_smoothing=0.1)
    assert torch.allclose(label_smoothed_ce(logits, t, 0.1), ref), "label smoothing does not match PyTorch"
    print("PASS")
_check1()

**Written (1a):** Show on paper that minimising label-smoothed CE equals minimising `(1-ε)·CE(onehot) + ε·CE(uniform)`. What does the second term do to the logits, in words?

**Your answer:**

## Exercise 2: Forward vs reverse KL (mode covering vs mode seeking)

Target `p` is a 1-D mixture: `0.5·N(-3, 0.7²) + 0.5·N(+3, 0.7²)`. Fit a single Gaussian `q = N(μ, σ²)`:
- `fit_forward()` minimises `KL(p‖q)`. Derive the closed-form answer (moment matching) and return `(mu, sigma)`.
- `fit_reverse(mu0, sigma0)` minimises `KL(q‖p)` by gradient descent on `(μ, log σ)` using numerical integration on a grid (`torch.linspace(-12, 12, 4001)`) or the reparameterisation trick. Start from the given initial point.

Then plot both fits over `p`.

In [ ]:
def log_p(x):
    return torch.logsumexp(torch.stack([math.log(.5) + torch.distributions.Normal(-3., .7).log_prob(x),
                                        math.log(.5) + torch.distributions.Normal(3., .7).log_prob(x)]), 0)

def fit_forward():
    # YOUR CODE: return (mu, sigma) as floats
    raise NotImplementedError

def fit_reverse(mu0, sigma0, steps=3000, lr=0.05):
    # YOUR CODE: return (mu, sigma) as floats
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    mu, s = fit_forward()
    assert abs(mu) < 1e-6 and abs(s - math.sqrt(9.49)) < 1e-6, f"forward KL fit wrong: {mu}, {s}"
    mu_r, s_r = fit_reverse(0.5, 1.0)
    assert abs(abs(mu_r) - 3) < 0.05 and abs(s_r - 0.7) < 0.05, f"reverse KL should lock onto one mode: {mu_r:.3f}, {s_r:.3f}"
    mu_l, _ = fit_reverse(-0.5, 1.0)
    assert mu_l * mu_r < 0, "different starting sides should find different modes"
    print(f"PASS  forward: N({mu:.2f}, {s:.2f}^2)   reverse: N({mu_r:.2f}, {s_r:.2f}^2)")
_check2()

**Written (2a):** RLHF and GRPO penalise `KL(π_θ ‖ π_ref)`. Which direction is that, and what does your experiment predict about how the policy can move relative to the reference model?

**Written (2b):** Maximum-likelihood training of a language model minimises which KL direction between data and model? Connect this to why LMs put probability mass on low-quality text.

**Your answer:**

## Exercise 3: Sampling from logits

Implement logit processors that return **new logits** with removed tokens set to `-inf` (this is how Hugging Face implements them):
- `apply_temperature(logits, T)`
- `top_k_filter(logits, k)`
- `top_p_filter(logits, p)`: keep the smallest set of highest-probability tokens whose cumulative probability is **≥ p**. Always keep at least one token. Work on batched `(B, V)` input without Python loops over the batch.

Then `sampling_entropy(logits, T)` returns the entropy of `softmax(logits / T)`.

In [ ]:
def apply_temperature(logits, T):
    # YOUR CODE
    raise NotImplementedError
def top_k_filter(logits, k):
    # YOUR CODE
    raise NotImplementedError
def top_p_filter(logits, p):
    # YOUR CODE
    raise NotImplementedError
def sampling_entropy(logits, T):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    L = torch.randn(16, 1000) * 3
    f = top_k_filter(L, 50)
    assert (torch.isfinite(f).sum(-1) == 50).all(), "top-k must keep exactly k (no ties here)"
    for p in (0.1, 0.5, 0.9, 0.999):
        f = top_p_filter(L, p); keep = torch.isfinite(f)
        probs = torch.softmax(L, -1)
        kept_mass = (probs * keep).sum(-1)
        assert (kept_mass >= p - 1e-12).all(), "kept set has less than p mass"
        smallest_kept = torch.where(keep, probs, torch.full_like(probs, 2.)).min(-1).values
        assert ((kept_mass - smallest_kept) < p).all(), "kept set is not minimal"
        biggest_dropped = torch.where(keep, torch.zeros_like(probs), probs).max(-1).values
        assert (biggest_dropped <= smallest_kept).all(), "must keep the highest-probability tokens"
        assert torch.equal(f[keep], L[keep]), "kept logits must be unchanged"
    one = torch.tensor([[10., 0., 0.]])
    assert torch.isfinite(top_p_filter(one, 0.01)).sum() == 1
    Ts = [0.25, 0.5, 1.0, 2.0, 4.0]; H = [float(sampling_entropy(L[:1], T)) for T in Ts]
    assert all(a < b for a, b in zip(H, H[1:])), "entropy must increase with temperature"
    assert abs(sampling_entropy(L[:1], 1e6) - math.log(1000)) < 1e-3
    print("PASS  entropy vs T:", [round(h, 2) for h in H])
_check3()

## Exercise 4: Is model B actually better? (a real eval decision)

Two models answered the same 500 questions. Model B scores about 2 points higher than model A (exact numbers are printed by the next cell).

Implement:
- `paired_bootstrap_ci(a, b, n_boot, alpha, seed)`: 95% percentile CI for `mean(b) - mean(a)`, resampling **questions** (pairs), not each model separately.
- `mcnemar_exact_p(a, b)`: two-sided exact McNemar p-value from the discordant pairs (use `scipy.stats.binomtest` or write the binomial sum yourself).
- `unpaired_ci(a, b)`: the naive normal-approximation CI that treats the two accuracies as independent.

Then write the paragraph you would send your team lead.

In [ ]:
g = torch.Generator().manual_seed(42)
latent = torch.rand(500, generator=g)                     # question difficulty, shared by both models
a = (latent < 0.712).long()
b = a.clone(); flip_up = torch.where(a == 0)[0][:20]; flip_dn = torch.where(a == 1)[0][:11]
b[flip_up] = 1; b[flip_dn] = 0
print(f"A={a.float().mean():.3f}  B={b.float().mean():.3f}")

In [ ]:
def paired_bootstrap_ci(a, b, n_boot=10000, alpha=0.05, seed=0):
    # YOUR CODE: return (lo, hi)
    raise NotImplementedError
def mcnemar_exact_p(a, b):
    # YOUR CODE
    raise NotImplementedError
def unpaired_ci(a, b, z=1.96):
    # YOUR CODE: return (lo, hi)
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    lo, hi = paired_bootstrap_ci(a, b)
    d = float(b.double().mean() - a.double().mean())
    assert lo < d < hi
    n01 = int(((a == 0) & (b == 1)).sum()); n10 = int(((a == 1) & (b == 0)).sum())
    se_paired = math.sqrt(n01 + n10 - (n01 - n10) ** 2 / 500) / 500
    assert abs((hi - lo) / (2 * 1.96 * se_paired) - 1) < 0.15, "paired CI width off; are you resampling pairs?"
    p = mcnemar_exact_p(a, b)
    k, m = min(n01, n10), n01 + n10
    ref = min(1.0, 2 * sum(math.comb(m, i) for i in range(k + 1)) / 2 ** m)
    assert abs(p - ref) < 1e-6, f"McNemar p looks wrong: {p} (expected {ref:.4f})"
    ulo, uhi = unpaired_ci(a, b)
    assert (uhi - ulo) > 2 * (hi - lo), "the unpaired CI should be much wider; check your formula"
    print(f"PASS  diff={d:+.3f}  paired 95% CI=({lo:+.3f}, {hi:+.3f})  McNemar p={p:.3f}  unpaired CI=({ulo:+.3f}, {uhi:+.3f})")
_check4()

**Written (4a):** Your paragraph to the team lead (5 sentences max): is B better, how sure are you, and what would you do next?

**Written (4b):** Why is the paired CI so much narrower than the unpaired one? When would they be about the same?

## Checkpoint

Without notes, say which probabilistic model each loss is the negative log-likelihood of: MSE, BCE, label-smoothed CE, and the DPO loss (hint for DPO: a Bradley-Terry model over preference pairs). If you can't, rewatch Stat 110 lectures on MLE and reread MML 8.3.

---
### Hints for self-grading
- 1a: the uniform term pushes all logits toward each other, capping confidence (a max-entropy regulariser).
- 2a: KL(π‖π_ref) is reverse KL from the policy's view: the policy may drop modes of the reference (mode seeking) but is punished heavily for putting mass where π_ref has almost none.
- 2b: MLE minimises forward KL(data‖model), which is mass-covering, so the model spreads probability over everything in the data, including bad text.
- 4a: the difference is not significant at the usual 5% level (the CI includes 0, p ≈ 0.15). Get more questions or more samples per question before deciding.
- 4b: both models get the same questions right or wrong most of the time, so paired differences cancel shared difficulty. They match when the models' errors are uncorrelated.